In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

In [2]:
# And metadata
md = pd.read_csv('../data/plasma_metadata_matched_all_outcomes.tsv', sep = '\t', dtype={'UCSD': str, 'Unnamed: 0': str, 'sample_name': str}).set_index('sample_name')

In [3]:
md['site'].isna().sum()

62

In [4]:
md['Race: White'] = md['RACE'].replace({2: 0, 3: 0, 5: 0, 50:0})

In [5]:
md['Race: Black or African American'] = md['RACE'].replace({2:1, 1: 0, 3: 0, 5: 0, 50:0})

In [6]:
md['Race: Black or African American'].value_counts()

0.0    411
1.0     95
Name: Race: Black or African American, dtype: int64

In [7]:
md['Race: Other'] = md['RACE'].replace({1:0, 2:0, 3: 1, 5: 1, 50:1})

In [8]:
md['Ethnicity: Hispanic'] = md['HISPANIC'].replace({9:0})

In [9]:
md['Ethnicity: Hispanic'].value_counts()

0.0    482
1.0     24
Name: Ethnicity: Hispanic, dtype: int64

In [10]:
# md_clean[['Race: White', 'Race: Black or African American', 'Race: Other', 'Ethnicity: Hispanic']] = md[['Race: White', 'Race: Black or African American', 'Race: Other', 'Ethnicity: Hispanic']]

In [11]:
md['Sex: Female'] = md['SEX'].replace({1: 0, 2: 1})

In [12]:
md['Sex: Female'].value_counts()

1.0    313
0.0    193
Name: Sex: Female, dtype: int64

In [13]:
md['Sex: Female'] = md['SEX'].replace({1: 0, 2: 1})

In [14]:
md['Age'] = md['NACCAGE']
md['BMI'] = md['NACCBMI']

In [15]:
md['Age'].min()

49.0

In [16]:
md['Age'].std()

7.451983651947449

In [17]:
md['BMI'] = md['BMI'].replace(np.inf, np.nan)

In [18]:
def make_group_summary_table(
    df: pd.DataFrame,
    group_col: str,
    variables: list,
    var_types: dict,
    group_order: list = None,
    continuous_test: str = "anova",  # "anova" or "kruskal"
    digits_mean: int = 3,
    digits_sd: int = 3,
    digits_pct: int = 1,
    digits_stat: int = 3,
    digits_p: int = 3,
):
    """
    Build a Table 1 by group with stats and p-values.

    Parameters
    ----------
    df : DataFrame
        Input data (one row per participant).
    group_col : str
        Column giving group (group).
    variables : list[str]
        Variables to summarize, in the order you want them displayed.
        Include a special string "#N" to insert the "No. of participants" row.
    var_types : dict[str, str]
        Map of variable -> "continuous" or "categorical".
        (For binary categorical, just ensure the column is categorical/boolean or has up to 2 levels.)
    group_order : list[str], optional
        Order of countries (columns). If None, inferred from the data.
    continuous_test : {"anova","kruskal"}
        Which test for continuous variables.
    digits_* : int
        Formatting precision.

    Returns
    -------
    pd.DataFrame
        Wide table with columns: Parameter, <group1>, <group2>, ..., Statistic, P-value
    """
    d = df.copy()
    # ensure group is categorical with desired order
    if group_order is None:
        group_order = list(pd.Series(d[group_col]).dropna().unique())
    d[group_col] = pd.Categorical(d[group_col], categories=group_order, ordered=True)

    # helper formatters
    def fmt_mean_sd(x):
        return f"{np.nanmean(x):.{digits_mean}f} ± {np.nanstd(x, ddof=1):.{digits_sd}f}"

    def fmt_n_pct(n, denom):
        pct = 100 * (n / denom) if denom > 0 else np.nan
        return f"{int(n)} ({pct:.{digits_pct}f}%)"

    def fmt_stat_p(stat, p):
        stat_s = f"{stat:.{digits_stat}f}" if pd.notnull(stat) else ""
        if p is None or np.isnan(p):
            p_s = ""
        elif p < 10**(-digits_p):
            p_s = f"<1e-{digits_p}"
        else:
            p_s = f"{p:.{digits_p}g}"
        return stat_s, p_s

    rows = []
    raw_pvals = []        # <--- COLLECT RAW P-VALUES HERE
    row_indices = []      # <--- KEEP TRACK OF WHICH ROW THEY BELONG TO
    
    # Top row: number of participants per group
    if "#N" in variables:
        total_n = len(d)
        row = {"Parameter": "No. of participants"}
        for c in group_order:
            n_c = (d[group_col] == c).sum()
            row[c] = fmt_n_pct(n_c, total_n)
        row["Statistic"], row["P-value"] = "", ""
        rows.append(row)

    for var in variables:
        if var == "#N":
            continue

        vtype = var_types.get(var, "categorical")  # default to categorical
        row = {"Parameter": var}

        # group-wise subsets
        groups = [d.loc[d[group_col] == c, var] for c in group_order]

        if vtype == "continuous":
            # per-group mean ± SD using non-missing n within each group
            for c, g in zip(group_order, groups):
                row[c] = fmt_mean_sd(g.astype(float))

            # test across countries
            if continuous_test == "anova":
                # f_oneway requires at least 2 non-empty groups
                non_empty = [g.dropna().astype(float) for g in groups if g.dropna().shape[0] > 1]
                if len(non_empty) >= 2:
                    stat, p = stats.f_oneway(*non_empty)
                else:
                    stat, p = np.nan, np.nan
                stat_s, p_s = fmt_stat_p(stat, p)
                row["Statistic"] = stat_s
                row["P-value"] = p_s
                
                # store raw p for correction
                raw_pvals.append(p)
                row_indices.append(len(rows))   # store row index
                rows.append(row)
                
            else:
                # Kruskal–Wallis
                non_empty = [g.dropna().astype(float) for g in groups if g.dropna().shape[0] > 0]
                if len(non_empty) >= 2:
                    stat, p = stats.kruskal(*non_empty)
                else:
                    stat, p = np.nan, np.nan
                stat_s, p_s = fmt_stat_p(stat, p)
                row["Statistic"] = stat_s
                row["P-value"] = p_s
                
                # store raw p for correction
                raw_pvals.append(p)
                row_indices.append(len(rows))   # store row index
                rows.append(row)

        else:
            # categorical (multi- or binary)
            # For each group, show n (%) where var is not NA (for denom) and for *all* levels combined.
            # If binary -> display count of the "positive" (second) level; if multi -> display total non-missing n (% of table N).
            gcat = d[[group_col, var]].copy()
            # Determine binary vs multi
            levels = gcat[var].dropna().unique()
            if gcat[var].dtype == "bool" or (len(levels) <= 2):
                # choose the "positive" level: True if boolean, else the 2nd sorted level
                if gcat[var].dtype == "bool":
                    pos_mask = gcat[var] == True
                    pos_label = "Yes"
                else:
                    lvls_sorted = pd.Series(levels).sort_values().tolist()
                    pos = lvls_sorted[-1] if len(lvls_sorted) == 2 else lvls_sorted[0]
                    pos_mask = gcat[var] == pos
                    pos_label = str(pos)

                # per-group: n_pos (% of non-missing in that group)
                for c in group_order:
                    sub = gcat.loc[gcat[group_col] == c, var]
                    denom = sub.notna().sum()
                    n_pos = (sub == (True if gcat[var].dtype == "bool" else pos)).sum()
                    row[c] = fmt_n_pct(n_pos, denom if denom > 0 else np.nan)

                # χ² test on contingency (countries x pos/neg)
                ct = pd.crosstab(gcat[group_col], pos_mask)
                if ct.shape[0] > 1 and ct.shape[1] > 1:
                    stat, p, _, _ = stats.chi2_contingency(ct)
                else:
                    stat, p = np.nan, np.nan
                stat_s, p_s = fmt_stat_p(stat, p)
                row["Statistic"] = stat_s
                row["P-value"] = p_s
                
                # store raw p for correction
                raw_pvals.append(p)
                row_indices.append(len(rows))   # store row index
                rows.append(row)

                # rename parameter to include the positive level 
                row["Parameter"] = f"{var}"  

            else:
                # Multi-category: show total non-missing n (%) per group relative to table N
                total_n = len(d)
                for c in group_order:
                    sub = gcat.loc[gcat[group_col] == c, var]
                    denom = sub.notna().sum()
                    row[c] = fmt_n_pct(denom, total_n)

                # χ² test across all levels
                ct = pd.crosstab(gcat[group_col], gcat[var])
                if ct.shape[0] > 1 and ct.shape[1] > 1:
                    stat, p, _, _ = stats.chi2_contingency(ct)
                else:
                    stat, p = np.nan, np.nan
                stat_s, p_s = fmt_stat_p(stat, p)
                row["Statistic"] = stat_s
                row["P-value"] = p_s
                
                # store raw p for correction
                raw_pvals.append(p)
                row_indices.append(len(rows))   # store row index
                rows.append(row)

    # ------------------------------
    #   Apply Benjamini–Hochberg FDR
    # ------------------------------
    raw_pvals_clean = [p if p is not None and not np.isnan(p) else 1.0 
                       for p in raw_pvals]

    reject, qvals, _, _ = multipletests(raw_pvals_clean, method="fdr_bh")

    # insert q-values back into rows
    for idx, q in zip(row_indices, qvals):
        rows[idx]["Q-value"] = f"{q:.{digits_p}g}"

    # Build final DataFrame
    cols = ["Parameter"] + group_order + ["Statistic", "Q-value"]
    out = pd.DataFrame(rows)[cols]

    return out


In [19]:
md['Diagonsis'].value_counts()

Cognitively Unimpaired    366
Cognitively Impaired      139
Name: Diagonsis, dtype: int64

In [20]:
variables = ["#N", 'Age', 'BMI', 'Sex: Female', 'Race: White', 'Race: Black or African American', 'Race: Other', 'Ethnicity: Hispanic', 'MOCA', 'Depression', 'Anxiety', 'Antidepressants',
'pTau181', 'pTau217', 'Abeta40', 'Abeta42', 'HEI2015Score']

var_types = {
    "Age": "continuous",
    "BMI": "continuous",
    "Sex: Female": "categorical",
    'Race: White': "categorical", 
    'Race: Black or African American': "categorical", 
    'Race: Other': "categorical", 
    'Ethnicity: Hispanic': "categorical",
    'MOCA': "continuous", 
    'Depression':"categorical", 
    'Antibiotic History: Within the Last Month':"categorical", 
    'Anxiety': "categorical",
    'Antidepressants': "categorical", 
    'pTau181': "continuous", 
    'pTau217': "continuous",
    'Abeta40': "continuous",
    'Abeta42': "continuous",
    'HEI2015Score': "continuous"
}

table1 = make_group_summary_table(
    md,
    group_col="Diagonsis",
    variables=variables,
    var_types=var_types,
    group_order=["Cognitively Unimpaired", "Cognitively Impaired"],
    continuous_test="kruskal",  
    digits_mean=1, digits_sd=1, digits_pct=1, digits_stat=2, digits_p=2
)


In [33]:
md[list(table1['Parameter'][1:])].describe().T

,count,mean,std,min,25%,50%,75%,max
Age,505.0,72.435644,7.451984,49.000000,68.00000,73.000000,77.000000,98.000000
BMI,481.0,27.227859,5.420637,15.100000,23.40000,26.500000,30.500000,54.700000
Sex: Female,506.0,0.618577,0.486217,0.000000,0.00000,1.000000,1.000000,1.000000
Race: White,506.0,0.792490,0.405925,0.000000,1.00000,1.000000,1.000000,1.000000
Race: Black or African American,506.0,0.187747,0.390896,0.000000,0.00000,0.000000,0.000000,1.000000
Race: Other,506.0,0.019763,0.139322,0.000000,0.00000,0.000000,0.000000,1.000000
Ethnicity: Hispanic,506.0,0.047431,0.212769,0.000000,0.00000,0.000000,0.000000,1.000000
MOCA,491.0,25.164969,4.332044,3.000000,23.00000,26.000000,28.000000,30.000000
Depression,506.0,0.065217,0.247153,0.000000,0.00000,0.000000,0.000000,1.000000
Anxiety,506.0,0.049407,0.216931,0.000000,0.00000,0.000000,0.000000,1.000000


In [21]:
md

,BAKER,FIEHN,METABOLON,NIGHTINGALE,UCSD,WISHART,TUULIA,SampleYear,NACC.ID,Follow up sample,...,HEI2015_Saturated_Fat,HEI2015_Added_Sugars,fiber,Race: White,Race: Black or African American,Race: Other,Ethnicity: Hispanic,Sex: Female,Age,BMI
sample_name,,,,,,,,,,,,,,,,,,,,,
15448.42368322,7.513676e+09,NaN,7.513676e+09,7.513676e+09,7513675899.0,NaN,NaN,NaN,NACC000806,NO,...,NaN,NaN,NaN,0.0,1.0,0.0,0.0,0.0,67.0,NaN
15448.42209258,7.513682e+09,7.513682e+09,7.513682e+09,7.513682e+09,7513681557.0,7.513682e+09,7.513682e+09,2022.0,NACC002325,YES,...,3.421491,9.186920,39.856434,1.0,0.0,0.0,0.0,1.0,80.0,21.9
15448.42385333,7.513668e+09,7.513663e+09,7.513613e+09,7.513613e+09,7513662591.0,7.513663e+09,7.513668e+09,2022.0,NACC002809,NO,...,0.000000,7.455456,14.781425,1.0,0.0,0.0,0.0,1.0,56.0,30.6
15448.42372244,7.513615e+09,7.513615e+09,7.513615e+09,7.513615e+09,7513687351.0,7.513615e+09,7.513676e+09,2022.0,NACC003007,NO,...,NaN,NaN,NaN,0.0,0.0,1.0,0.0,1.0,72.0,22.7
15448.42326795,7.513682e+09,7.513682e+09,7.513682e+09,7.513682e+09,7513614766.0,7.513682e+09,7.513682e+09,2023.0,NACC004873,NO,...,NaN,NaN,NaN,1.0,0.0,0.0,0.0,0.0,63.0,33.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15448.0042121186,7.513671e+09,5.003104e+09,7.513671e+09,7.513671e+09,5003103634.0,5.003104e+09,7.513671e+09,2022.0,NACC993291,NO,...,7.242449,10.000000,85.491783,1.0,0.0,0.0,0.0,0.0,72.0,23.6
15448.42556374,7.513615e+09,7.513615e+09,7.513615e+09,7.513615e+09,7513615062.0,7.513615e+09,7.513615e+09,2022.0,NACC994923,NO,...,5.021415,9.820618,31.642000,1.0,0.0,0.0,0.0,0.0,81.0,NaN
15448.42593049,7.513682e+09,7.513682e+09,7.513682e+09,7.513682e+09,7513681539.0,7.513682e+09,7.513595e+09,2022.0,NACC997037,NO,...,NaN,NaN,NaN,1.0,0.0,0.0,0.0,1.0,60.0,24.4


In [21]:
table1.style

,Parameter,Cognitively Unimpaired,Cognitively Impaired,Statistic,Q-value
0,No. of participants,366 (72.3%),139 (27.5%),,nan
1,Age,71.9 ± 7.1,73.8 ± 8.1,6.50,0.043
2,BMI,27.2 ± 5.3,27.2 ± 5.2,0.00,1
3,Sex: Female,234 (63.9%),78 (56.1%),2.29,0.26
4,Race: White,294 (80.3%),106 (76.3%),0.78,0.52
5,Race: Black or African American,65 (17.8%),30 (21.6%),0.73,0.52
6,Race: Other,7 (1.9%),3 (2.2%),0.00,1
7,Ethnicity: Hispanic,13 (3.6%),11 (7.9%),3.33,0.22
8,MOCA,26.8 ± 2.4,20.8 ± 5.2,168.11,3.1e-37
9,Depression,20 (5.5%),13 (9.4%),1.90,0.3


In [55]:
# Save or display
table1.to_csv("demographics_table.csv", index=False)